<a href="https://colab.research.google.com/github/aditya0589/notebooks/blob/main/ML%20System%20Design/ch_3_training_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **CH-3: Training Data**


ML curricula are heavily skewed towards modeling, which is considered by many
researchers and engineers as the “fun” part of the process. Building a state-of-the-art model is interesting.
Spending days wrangling with a massive amount of malformatted data that doesn’t even fit into your machine’s
memory is frustrating.

<br>
----------------------------------------------

<br>

### **Sampling**

Sampling is an integral part of the ML workflow that is, unfortunately, often overlooked in typical ML
coursework.

Sampling happens in many steps of an ML project lifecycle, such as sampling from all possible
real-world data to create training data, sampling from a given dataset to create splits for training, validation, and
testing, or sampling from all possible events that happen within your ML system for monitoring purposes.

There are two main types of sampling. They are:

1. Non-Probability sampling
2. Probability sampling

<br>

### **Non Probabilistic Sampling**

Non-probability sampling is when the selection of data isn’t based on any probability criteria. Here are some of
the criteria for non-probability sampling.

**Convenience sampling**: samples of data are selected based on their availability. This sampling method
is popular because, well, it’s convenient.

**Snowball sampling**: future samples are selected based on existing samples. For example, to scrape
legitimate Twitter accounts without having access to Twitter databases, you start with a small number
of accounts then you scrape all the accounts in their following, and so on.

**Judgment sampling**: experts decide what samples to include.

**Quota sampling**: you select samples based on quotas for certain slices of data without any
randomization.

For example, when doing a survey, you might want 100 responses from each of the age
groups: under 30 years old, between 30 and 60 years old, above 50 years old, regardless of the actual
age distribution in the real world.

Non-probability sampling can be a quick and easy way to gather your initial data to get your project off the
ground. However, for reliable models, you might want to use probability-based sampling

<br>

### **Probability Sampling**

1. **Simple Random Sampling**

In the simplest form of random sampling, you give all samples in the population equal probabilities of being
selected. For example, you randomly select 10% of all samples, giving all samples an equal 10% chance of
being selected.

The advantage of this method is that it’s easy to implement. The drawback is that rare categories of data might
not appear in your selection. Consider the case where a class appears only in 0.01% of your data population.

If
you randomly select 1% of your data, samples of this rare class will unlikely be selected. Models trained on this
selection might think that this rare class doesn’t exist.



In [1]:
import random

population = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

sample = random.sample(population, 3)

print(sample)

[9, 3, 10]


In [2]:
#sampling with replacement
import numpy as np

population = [1, 2, 3, 4, 5]

sample = np.random.choice(population, size=3, replace=True)

print(sample)

[3 1 3]


In [3]:
#sampling without replacement
import numpy as np

population = [1, 2, 3, 4, 5]

sample = np.random.choice(population, size=3, replace=False)

print(sample)

[5 4 3]


### **Stratified Sampling**

To avoid the drawback of simple random sampling listed above, you can first divide your population into the
groups that you care about and sample from each group separately.

For example, to sample 1% of data that has
two classes A and B, you can sample 1% of class A and 1% of class B. This way, no matter how rare class A or
B is, you’ll ensure that samples from it will be included in the selection. Each group is called a **strata**, and this
method is called **stratified sampling**.


One drawback of this sampling method is that it isn’t always possible, such as when it’s impossible to divide all
samples into groups.

This is especially challenging when one sample might belong to multiple groups as in the
case of multilabel tasks . For instance, a sample can be both class A and class B.


In [4]:
import pandas as pd

df = pd.DataFrame({
    'student_id': range(1, 11),
    'gender': ['M', 'F', 'M', 'M', 'F',
               'F', 'M', 'F', 'M', 'F'],
    'marks': [85, 90, 78, 88, 92,
              75, 81, 95, 87, 80]
})

print(df)

   student_id gender  marks
0           1      M     85
1           2      F     90
2           3      M     78
3           4      M     88
4           5      F     92
5           6      F     75
6           7      M     81
7           8      F     95
8           9      M     87
9          10      F     80


In [5]:
# method 1: using pandas aggregations
sample_df = (
    df.groupby('gender', group_keys=False)
      .apply(lambda x: x.sample(frac=0.5, random_state=42))
)

print(sample_df)

   student_id gender  marks
4           5      F     92
9          10      F     80
2           3      M     78
8           9      M     87


/tmp/ipykernel_1541/1054712198.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(frac=0.5, random_state=42))


In [8]:
# method2: using scikit-learn
from sklearn.model_selection import train_test_split

train, test = train_test_split(
    df,
    test_size=0.2,
    stratify=df['gender'],
    random_state=42
)

print(train)

   student_id gender  marks
6           7      M     81
9          10      F     80
4           5      F     92
1           2      F     90
2           3      M     78
0           1      M     85
3           4      M     88
5           6      F     75


### **Weighted Sampling**

In weighted sampling, each sample is given a weight, which determines the probability of it being selected. For
example, if you have three samples A, B, C and want them to be selected with the probabilities of 50%, 30%,
20% respectively, you can give them the weights 0.5, 0.3, 0.2.


This method allows you to leverage domain expertise. For example, if you know that a certain subpopulation of
data, such as more recent data, is more valuable to your model and want it to have a higher chance of being
selected, you can give it a higher weight.

In [9]:
random.choices(population=[1, 2, 3, 4, 100, 1000],
 weights=[0.2, 0.2, 0.2, 0.2, 0.1, 0.1],
 k=2)


[3, 3]

### **Importance Sampling**

Importance sampling is one of the most important sampling methods, not just in ML. It allows us to sample
from a distribution when we only have access to another distribution.


Imagine you have to sample x from a distribution P(x), but P(x) is really expensive, slow, or infeasible to
sample from. However, you have a distribution Q(x) that is a lot easier to sample from. So you sample x from
Q(x) instead and weight this sample by P(x)
Q(x) Q(x) is called the proposal distribution or the importance
distribution. Q(x) can be any distribution as long as Q(x) > 0 whenever P(x) ≠ 0. The equation below shows
that in expectation, x sampled from P(X) is equal to x sampled from Q(x) weighted by P(x)
Q(x)


.
$$
\mathbb{E}_{P(x)}[x]
=
\sum_x P(x)x
=
\sum_x Q(x)x \frac{P(x)}{Q(x)}
=
\mathbb{E}_{Q(x)}
\left[
x \frac{P(x)}{Q(x)}
\right]
$$



---------------------------------------------

<br>

### **Labelling**

**1. Hand labelling**:

hand-labeling data can be expensive,
especially if subject matter expertise is required.

To classify whether a comment is spam, you might be able
to find 200 annotators on a crowdsourcing platform and train them in 15 minutes to label your data. However, if
you want to label chest X-rays, you’d need to find board-certified radiologists, whose time is limited and
expensive.

Hand labeling poses a **threat to data privacy**. Hand labeling means that someone has to look at your
data, which isn’t always possible if your data has strict privacy requirements.

hand labeling is slow. For example, accurate transcription of speech utterance at phonetic level can take
400 times longer than the utterance duration.

**Label Multiplicity**:

Often, to obtain enough labeled data, companies have to use data from multiple sources and rely on multiple
annotators who have different levels of expertise. These different data sources and annotators also have
different levels of accuracy. This leads to the problem of label ambiguity or label multiplicity.

**Data Lineage**

Most of the times, the problem of poor model accuracy can be solved by feeding it more data.

However sometimes, feeding a model a lot of data can lead to degradation in its performance. This is because the new data might be poorly labelled and not be on par with the needed data quality.

Therefore keeping track of the origin of data is helpful for debugging and understanding.

**Semi-Supervised Learning**

Uses a small amount of labeled data and a large amount of unlabeled data to train a model.

Example:

1,000 cat/dog images are labeled.
100,000 images are unlabeled.
The model learns from both.

Goal: Reduce the need for expensive manual labeling.

**Weak Supervised Learning**

Uses inaccurate, noisy, or coarse-grained labels instead of perfectly labeled data.

Example:

An image is labeled "contains a dog" but the dog's exact location is not provided.
Labels are generated automatically and may contain errors.

Goal: Leverage cheap but imperfect labels to train models.
